In [2]:
# Aniesh Metric
import numpy as np
def compute_accuracy_from_confusion_matrix(cm):
    # True Positives (TP): Diagonal elements
    true_positive = np.diag(cm)

    # False Positives (FP): Column sum - TP
    false_positive = cm.sum(axis=0) - true_positive

    # False Negatives (FN): Row sum - TP
    false_negative = cm.sum(axis=1) - true_positive

    # True Negatives (TN): Total sum - (TP + FP + FN)
    total = cm.sum()
    true_negative = total - (true_positive + false_positive + false_negative)

    # Debug prints for each class
    for i in range(len(true_positive)):

        if true_negative[i] + false_positive[i] == 0:
            print(f"  ⚠️ No negative examples for class {i}")
        if true_positive[i] + false_negative[i] == 0:
            print(f"  ⚠️ No positive examples for class {i}")
        if true_negative[i] == 0 and false_positive[i] > 0:
            print(f"Class {i}:")
            print(f"  TP = {true_positive[i]}, FP = {false_positive[i]}, FN = {false_negative[i]}, TN = {true_negative[i]}")
            print(f"  ⚠️ Specificity for class {i} is 0 — all negative samples predicted as class {i}")
            print("\nFull Confusion Matrix:\n", cm)

    # Accuracy: Total correct predictions / Total samples
    accuracy = true_positive.sum() / total

    # Sensitivity (Recall): TP / (TP + FN)
    with np.errstate(divide='ignore', invalid='ignore'):
        sensitivity = np.divide(true_positive, true_positive + false_negative)
        sensitivity[np.isnan(sensitivity)] = 0

    # Specificity: TN / (TN + FP)
    with np.errstate(divide='ignore', invalid='ignore'):
        specificity = np.divide(true_negative, true_negative + false_positive)
        specificity[np.isnan(specificity)] = 0

    # Balanced Accuracy: Average of Sensitivity and Specificity
    balanced_accuracy = np.mean((sensitivity + specificity) / 2)

    return accuracy, balanced_accuracy

def compute_additional_metrics_from_confusion_matrix(cm):
    """
    Compute specificity, sensitivity, positive predictive value (PPV),
    and negative predictive value (NPV) from a multi-class confusion matrix.

    :param cm: NumPy array representing the confusion matrix (square matrix).
    :return: Dictionary containing computed metrics for each class.
    """
    # True Positives (TP): Diagonal elements
    true_positive = np.diag(cm)

    # False Positives (FP): Column sum - TP
    false_positive = cm.sum(axis=0) - true_positive

    # False Negatives (FN): Row sum - TP
    false_negative = cm.sum(axis=1) - true_positive

    # True Negatives (TN): Total sum - (TP + FP + FN)
    total = cm.sum()
    true_negative = total - (true_positive + false_positive + false_negative)

    # Compute metrics
    sensitivity = true_positive / (true_positive + false_negative)  # Recall
    specificity = true_negative / (true_negative + false_positive)
    ppv = true_positive / (true_positive + false_positive)  # Precision
    npv = true_negative / (true_negative + false_negative)
    f1_score = 2 * ppv * sensitivity / (ppv + sensitivity)

    # Handle division by zero
    sensitivity = np.nan_to_num(sensitivity)
    specificity = np.nan_to_num(specificity)
    ppv = np.nan_to_num(ppv)
    npv = np.nan_to_num(npv)
    f1_score = np.nan_to_num(f1_score)

    # Return metrics for each class
    return {
        "sensitivity": sensitivity.tolist(),
        "specificity": specificity.tolist(),
        "positive_predictive_value": ppv.tolist(),
        "negative_predictive_value": npv.tolist(),
        "f1_score": f1_score.tolist(),
    }


In [3]:
import pandas as pd
from sklearn.metrics import confusion_matrix, balanced_accuracy_score

# Load data
path = '/niddk-data-central/leo_workspace/iWatch-Validation/W/CHAP-FT/predictions/i0195A.csv' # TODO: Use your path
df = pd.read_csv(path)

# Ensure the order and mapping
class_names = sorted(set(df['prediction']).union(set(df['label'])))
class_to_int = {name: i for i, name in enumerate(class_names)}

# Map to integer labels
y_pred = df['prediction'].map(class_to_int).values
y_true = df['label'].map(class_to_int).values

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))

# SKLearn metrics
print("Confusion matrix:\n", cm)

# Balanced accuracy
bal_acc = balanced_accuracy_score(y_true, y_pred)
print("Balanced accuracy:", bal_acc)

# Optionally, show with class labels
import pandas as pd
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("\nConfusion Matrix (with labels):")
print(cm_df)


Confusion matrix:
 [[  0  42]
 [ 99 405]]
Balanced accuracy: 0.4017857142857143

Confusion Matrix (with labels):
             not-sitting  sitting
not-sitting            0       42
sitting               99      405


In [ ]:
# animesh code
import numpy as np
compute_accuracy_from_confusion_matrix(cm)

Class 1:
  TP = 405, FP = 42, FN = 99, TN = 0
  ⚠️ Specificity for class 1 is 0 — all negative samples predicted as class 1

Full Confusion Matrix:
 [[  0  42]
 [ 99 405]]


(0.7417582417582418, 0.4017857142857143)